# Double DQN: раздвоить `max`, чтобы перестать переоценивать ценность

**Статья.** van Hasselt, Guez & Silver, *Deep Reinforcement Learning with Double Q-learning*, AAAI 2016 ([arXiv:1509.06461](https://arxiv.org/abs/1509.06461)), поверх DQN (Mnih et al., Nature 2015).

**Узел дерева:** `rl_dqn` (100 days · ветка RL/value-based).

**Что строим и обучаем.** Минимальный, но честный DQN на CartPole-v1 (`gymnasium`), и поверх него — одну строчку из статьи: TD-таргет Double DQN. Обучаем end-to-end обе руки абляции (vanilla vs Double), меряем возврат и — главное — *переоценку ценности* (predicted минус actual), ту самую величину с Fig.3 статьи. Плюс игрушечный демонстратор механизма с Fig.1.

**Компьют.** CPU, несколько минут на весь прогон (3 сида × 30k шагов на руку). GPU не нужен — гиперпараметры ужаты под CPU, но протокол метода не урезан.

**Parts.**
1. Эксперименты-первыми — что показала статья, до метода.
2. Зачем это: проблема переоценки и фишка раздвоения.
3. Теория Фейнмана: откуда берётся положительный сдвиг у `max`.
4. setup: модель, реплей, среда.
5. Part A — TD-таргет (ядро, вклад статьи).
6. Part B — диагностика переоценки `predicted − actual` (headline статьи).
7. Part C — игрушечный механизм Fig.1: single vs double оценщик.
8. Сборка абляции, ключевая фигура, контраст «пощупать фишку».
9. Рефлексия.

Ядро метода — твоё. Бойлерплейт (среда, сеть, реплей, training-loop, графики) дан готовым.

## Part 0. Эксперименты-первыми (Results читаем до метода)

Прежде чем лезть в код — три слота от руки. Открой abstract и Fig.3/Fig.6 статьи и ответь своими словами (это не оценивается, это калибровка ожиданий).

1. **Что показали против abstract.** Заявка: DQN систематически *переоценивает* Q, и на Atari это вредит. Что именно на их графиках поддерживает заявку — кривая value-estimate или кривая score? Чем мерили «переоценку»?

2. **Самый слабый эксперимент.** Где эффект наименее убедителен — на каких играх Double DQN почти не отличается от DQN? Что это говорит о том, когда фишка важна, а когда нет?

3. **Где сломалось / что осталось за кадром.** Double DQN меняет ровно одну вещь в таргете и больше ничего. Что НЕ чинит (исследование, сеть, реплей)? Где переоценка всё ещё может протечь?

Записывай прямо здесь, в этой ячейке.

## Part 1. Зачем это вообще

Q-learning бутстрапит: таргет для $Q(s,a)$ опирается на оценку ценности следующего состояния. В DQN эта оценка — $\max_{a'} Q_{\text{target}}(s', a')$. Проблема в том, что один и тот же `max` и **выбирает** действие, и **оценивает** его. А `max` по зашумлённым оценкам — смещённая операция: $\mathbb{E}[\max_a \hat Q(a)] \ge \max_a \mathbb{E}[\hat Q(a)]$. Шум, который случайно задрал какое-то $\hat Q(a')$ вверх, и выбирается, и тут же подставляется в таргет. Сдвиг — систематически вверх. Так Q раздувается, и раздувание просачивается обратно через бутстрап.

**Фишка Double DQN — раздвоить роли.** Пусть одна сеть **выбирает** действие, а другая, независимая, его **оценивает**. Шум, поднявший выбор у первой сети, у второй уже не скоррелирован с этим же выбором — и систематический сдвиг вверх исчезает. В DQN вторая сеть уже есть бесплатно: target-network. Так что вся правка — это какую сеть спросить про argmax. Одна строчка.

**Почему CartPole.** Канонический управляющий бенчмарк с настоящей динамикой среды, встроен в gymnasium, считается на CPU за минуты. Возврат ограничен сверху (500), что делает переоценку — predicted выше достижимого — наблюдаемой напрямую, без Atari-масштаба.

❓ **Inline.** target-network в обычном DQN вводили ради *стабильности* (чтобы цель не гналась за собой). Double DQN переиспользует её для другой цели. Почему наличие лагающей второй сети — это удачно именно для развязки select/evaluate, и что было бы, спроси мы argmax у самой target-сети?

## Part 2. Теория (Фейнман): откуда сдвиг и где у меня дыра

**Задача.** Дано состояние $s'$ и истинные ценности действий $Q^*(s',a')$. Нам нужна оценка $\max_{a'} Q^*(s',a')$ — ценность лучшего следующего хода, чтобы построить TD-таргет $y = r + \gamma \cdot (\text{эта оценка}) \cdot (1-\text{done})$.

**Механизм сдвига.** Мы знаем не $Q^*$, а зашумлённую оценку $\hat Q$. Возьмём для интуиции крайний случай: все истинные $Q^*(a')=0$, а оценки независимы, $\hat Q(a') \sim \mathcal N(0,1)$. Тогда
$$\mathbb{E}\big[\max_{a'} \hat Q(a')\big] > 0 = \max_{a'} \mathbb{E}[\hat Q(a')],$$
и разрыв растёт с числом действий (максимум из большего числа гауссиан в среднем больше). Это и есть переоценка: `max` по шуму смещён вверх.

**Развязка.** Возьмём два независимых набора оценок, $\hat Q_A$ и $\hat Q_B$. Выберем действие по $A$: $a^\star = \arg\max_{a'} \hat Q_A(a')$. Оценим его по $B$: $\hat Q_B(a^\star)$. Раз $B$ не участвовал в выборе, $\mathbb{E}[\hat Q_B(a^\star)] = 0$ — сдвиг ушёл. Это double-оценщик.

В deep-варианте $A$ = online-сеть (свежий argmax), $B$ = target-сеть (оценка):
$$y^{\text{DQN}} = r + \gamma\, \max_{a'} Q_{\text{target}}(s',a'), \qquad y^{\text{Double}} = r + \gamma\, Q_{\text{target}}\big(s',\, \arg\max_{a'} Q_{\text{online}}(s',a')\big).$$

**Что метод измеряет на практике.** *Переоценку ценности*: насколько то, что агент себе **предсказывает** ($\max_a Q(s_0,a)$), выше того, что он **реально получает** (Monte-Carlo возврат жадного роллаута из $s_0$). Положительная разница = переоценка.

**Где у меня дыры.** Три ядровые функции: (A) сам TD-таргет с флагом `double`; (B) диагностика `predicted − actual`; (C) игрушечный механизм Fig.1. Их и пишем.

❓ **Inline.** В формуле для $y^{\text{Double}}$ почему argmax считается по `online`, а не по `target`? Что сломается в развязке, если и выбор, и оценку отдать одной target-сети?

In [ ]:
# setup: депсы, среда, модель, реплей, training-loop, eval. Готовое, не трогаем.
# (если gymnasium не стоит:  pip install gymnasium numpy torch)
import copy, math, random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

DEVICE = torch.device("cpu")
ENV_ID = "CartPole-v1"
Transition = namedtuple("Transition", ("s", "a", "r", "s2", "done"))


class QNet(nn.Module):
    """MLP Q-network: state -> Q-value per action."""
    def __init__(self, obs_dim, n_act, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_act),
        )
    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    """Uniform experience replay: break temporal correlation by random sampling."""
    def __init__(self, cap):
        self.buf = deque(maxlen=cap)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, n):
        batch = random.sample(self.buf, n)
        b = Transition(*zip(*batch))
        s = torch.as_tensor(np.array(b.s), dtype=torch.float32, device=DEVICE)
        a = torch.as_tensor(b.a, dtype=torch.long, device=DEVICE).unsqueeze(1)
        r = torch.as_tensor(b.r, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        s2 = torch.as_tensor(np.array(b.s2), dtype=torch.float32, device=DEVICE)
        done = torch.as_tensor(b.done, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        return s, a, r, s2, done
    def __len__(self):
        return len(self.buf)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

print("setup ok:", ENV_ID, DEVICE)

## Part A. TD-таргет — ядро, вклад статьи

Вот единственное место, где Double DQN отличается от vanilla. Флаг `double` — единственный переключатель между руками абляции; всё остальное в обучении общее. Реализуй обе ветки.

Сигнатура и контракт ниже. Тело — твоё. Полный рецепт, если застрянешь, — в SPOILERS внизу.

In [ ]:
def td_target(reward, next_state, done, online, target, gamma, double):
    """Bootstrap TD target y for a minibatch — the quantity Q(s,a) is regressed toward.

    Two arms selected by `double`:
      double=False -> vanilla DQN (Mnih 2015): one network both picks and scores the
                      next action.
      double=True  -> Double DQN (van Hasselt 2016): the choice of the next action and
                      the scoring of it are decoupled across the two given networks.
    A terminal transition contributes no bootstrap term.

    Args:
        reward:     [B,1] float — immediate reward r.
        next_state: [B,obs_dim] float — s'.
        done:       [B,1] float in {0,1} — 1.0 iff s' is terminal.
        online:     QNet — the online (current) network.
        target:     QNet — the lagging target network.
        gamma:      float — discount.
        double:     bool — arm selector (see above).
    Returns:
        [B,1] float — the TD target y. No gradient should flow through it.
    Note: must not require grad through the target (it is a regression label).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# shapes + no-grad + terminal masking + the defining property of the two arms
probe = QNet(4, 3)
B = 16
rew = torch.zeros(B, 1)
ns = torch.randn(B, 4)
notdone = torch.zeros(B, 1)

y = td_target(rew, ns, notdone, probe, probe, 0.99, double=False)
assert y.shape == (B, 1), f"TD target must be [B,1], got {tuple(y.shape)}"
assert not y.requires_grad, "TD target is a label: no grad should flow through it"

# terminal -> pure reward, no bootstrap
yt = td_target(torch.ones(B, 1), ns, torch.ones(B, 1), probe, probe, 0.99, double=False)
assert torch.allclose(yt, torch.ones(B, 1)), "on terminal (done=1) target must equal reward only"

# When online==target, double picks argmax of the SAME net it evaluates -> identical to max.
# So with a shared net, double and vanilla MUST coincide exactly.
y_van = td_target(rew, ns, notdone, probe, probe, 0.99, double=False)
y_dbl = td_target(rew, ns, notdone, probe, probe, 0.99, double=True)
assert torch.allclose(y_van, y_dbl), \
    "with online==target the double target reduces to the max target (same net selects & evaluates)"

# With DIFFERENT nets the double target gathers target-Q at online's argmax, which is
# <= target's own max -> double target is never above vanilla target (per element).
on, tg = QNet(4, 3), QNet(4, 3)
ya = td_target(rew, ns, notdone, on, tg, 0.99, double=False)
yb = td_target(rew, ns, notdone, on, tg, 0.99, double=True)
assert (yb <= ya + 1e-5).all(), "double target gathers at online-argmax <= target-max, so y_double <= y_vanilla"
print("ok")

In [ ]:
# training driver + epsilon-greedy collection + hard target sync. Готовое.
# Снимает snapshot (online, target) в момент, когда target максимально отстаёт от
# online — реалистичное условие, при котором max-правило переоценивает.
def train(double, seed=0, total_steps=30000):
    set_seed(seed)
    env = gym.make(ENV_ID)
    env.reset(seed=seed); env.action_space.seed(seed)
    obs_dim = env.observation_space.shape[0]; n_act = env.action_space.n

    online = QNet(obs_dim, n_act).to(DEVICE)
    target = QNet(obs_dim, n_act).to(DEVICE)
    target.load_state_dict(online.state_dict())
    opt = torch.optim.Adam(online.parameters(), lr=1e-3)

    buf = ReplayBuffer(20000)
    gamma, batch_size, learn_start, target_update = 0.99, 64, 1000, 500
    eps_start, eps_end, eps_decay = 1.0, 0.02, 4000

    snapshot = None; step = 0
    s, _ = env.reset(seed=seed)
    while step < total_steps:
        eps = eps_end + (eps_start - eps_end) * math.exp(-step / eps_decay)
        if random.random() < eps:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = int(online(torch.as_tensor(s, dtype=torch.float32)).argmax().item())
        s2, r, term, trunc, _ = env.step(a)
        done = term  # time-limit truncation is NOT terminal for bootstrapping
        buf.push(s, a, r, s2, float(done))
        s = s2 if not (term or trunc) else env.reset()[0]
        step += 1

        if len(buf) >= learn_start:
            bs, ba, br, bs2, bdone = buf.sample(batch_size)
            y = td_target(br, bs2, bdone, online, target, gamma, double)
            q = online(bs).gather(1, ba)
            loss = F.smooth_l1_loss(q, y)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(online.parameters(), 10.0); opt.step()
            if step % target_update == target_update - 1:
                snapshot = (copy.deepcopy(online), copy.deepcopy(target))
            if step % target_update == 0:
                target.load_state_dict(online.state_dict())
    env.close()
    if snapshot is None:
        snapshot = (copy.deepcopy(online), copy.deepcopy(target))
    return online, target, snapshot


@torch.no_grad()
def eval_return(policy_fn, episodes=20, seed=1000):
    env = gym.make(ENV_ID); returns = []
    for i in range(episodes):
        s, _ = env.reset(seed=seed + i); done = False; ep = 0.0
        while not done:
            s, r, term, trunc, _ = env.step(policy_fn(s)); ep += r; done = term or trunc
        returns.append(ep)
    env.close()
    return float(np.mean(returns)), float(np.std(returns))


def greedy_policy(net):
    @torch.no_grad()
    def pi(s):
        return int(net(torch.as_tensor(s, dtype=torch.float32)).argmax().item())
    return pi


def random_policy(n_act=2):
    def pi(s):
        return random.randrange(n_act)
    return pi

print("train/eval helpers ready")

## Part B. Переоценка ценности — headline-величина статьи (Fig.3)

Важно: переоценка статьи — это НЕ конструктивный зазор «max против gather» внутри одной пары сетей. Это зазор между тем, что агент сам себе **предсказывает** в стартовом состоянии, и тем, что он **реально получает**, играя жадно из этого состояния. Положительное значение = агент думает, что ценнее, чем выходит на деле.

Тебе дан жадный роллаут (среда настоящая). Реализуй сбор `predicted` и `actual` и их разность.

In [ ]:
@torch.no_grad()
def value_overestimation(net, gamma=0.99, episodes=20, seed=3000):
    """Paper's headline diagnostic (van Hasselt 2016, Fig.3): how much the agent's OWN
    value estimate at the start state exceeds the value it ACTUALLY realises by acting
    greedily from that same start state.

    For each of `episodes` start states s0 (env reset with seed+i): form the agent's
    predicted value of s0, then play the greedy policy of `net` to termination and form
    the realised (discounted) return from s0. A positive mean gap is overestimation.

    Args:
        net:      QNet — evaluated greedily.
        gamma:    float — discount for the realised return.
        episodes: int — number of start states / rollouts.
        seed:     int — base env seed (episode i uses seed+i).
    Returns:
        (predicted_mean, actual_mean, gap) floats, gap = predicted_mean - actual_mean.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# A trained net should realise a sizeable return; gap = predicted - actual by definition.
# Use a quick-trained net (short budget just to exercise the function).
on_quick, _, _ = train(double=False, seed=0, total_steps=8000)
pm, am, gap = value_overestimation(on_quick, episodes=10)
assert isinstance(pm, float) and isinstance(am, float) and isinstance(gap, float), "must return three floats"
assert abs((pm - am) - gap) < 1e-6, "gap must equal predicted_mean - actual_mean"
assert am > 0.0, "a trained CartPole policy must realise positive discounted return"
# discounted return is bounded by 1/(1-gamma); predicted lives on the same value scale
assert am < 1.0 / (1 - 0.99) + 1.0, "discounted return cannot exceed 1/(1-gamma)"
print(f"predicted={pm:.2f} actual={am:.2f} gap={gap:+.2f}")
print("ok")

## Part B′. Конструктивный зазор max-vs-double на общей паре сетей

Это вспомогательная диагностика: на одной зафиксированной паре `(online, target)` и одном наборе состояний — насколько value по max-правилу выше value по double-правилу. По построению $\ge 0$. Она показывает сам сдвиг внутри таргета, в отрыве от обучения. Реализуй обе оценки ценности и их разность.

In [ ]:
@torch.no_grad()
def overestimation_gap(online, target, states):
    """On a fixed (online, target) pair and a fixed batch of states, the systematic
    upward bias the max rule injects into the bootstrap value relative to the decoupled
    (Double DQN) rule.

    Form two scalar value estimates averaged over `states`: the value the vanilla-DQN
    target would use, and the value the Double-DQN target would use on the SAME pair.
    Their difference is the bias and is >= 0 by construction.

    Args:
        online: QNet — selects the action in the decoupled estimate.
        target: QNet — scores the action in both estimates.
        states: [N,obs_dim] float — states to average over.
    Returns:
        (dqn_val, double_val, gap) floats, gap = dqn_val - double_val (>= 0).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
@torch.no_grad()
def collect_states(net, episodes=30, seed=2000):
    env = gym.make(ENV_ID); pi = greedy_policy(net); states = []
    for i in range(episodes):
        s, _ = env.reset(seed=seed + i); done = False
        while not done:
            states.append(s); s, r, term, trunc, _ = env.step(pi(s)); done = term or trunc
    env.close()
    return torch.as_tensor(np.array(states), dtype=torch.float32)

on, tg = QNet(4, 3), QNet(4, 3)
st = torch.randn(200, 4)
dv, ddv, gap = overestimation_gap(on, tg, st)
assert abs((dv - ddv) - gap) < 1e-6, "gap must equal dqn_val - double_val"
assert gap >= -1e-6, "max-rule value >= decoupled value by construction (gap >= 0)"
# with online==target the two estimates coincide -> gap == 0
dv2, ddv2, gap2 = overestimation_gap(on, on, st)
assert abs(gap2) < 1e-6, "with online==target the gap must be exactly 0 (same net selects & evaluates)"
print(f"dqn_val={dv:.3f} double_val={ddv:.3f} gap={gap:+.3f}")
print("ok")

## Part C. Игрушечный механизм Fig.1: single vs double оценщик

Чистая изоляция механизма — без RL, без сетей. Истинные ценности всех действий равны (скажем, 0), оценки — независимый гауссов шум. Покажи, что single (max) оценщик переоценивает и переоценка растёт с числом действий, а double оценщик несмещён.

Два независимых набора оценок: первым **выбираешь** действие, вторым его **оцениваешь**. Это и есть развязка из теории, но на чистом шуме.

In [ ]:
def overestimation_mechanism(n_actions=(2, 4, 8), trials=20000, seed=123):
    """Paper Sec.3 / Fig.1 mechanism in isolation. True Q*(a)=0 for every action;
    estimates are independent standard-normal noise. Contrast two estimators of
    max_a Q*(a) (whose true value is 0):
      - single estimator: one noisy table both picks and scores the action;
      - double estimator: one noisy table picks the action, a SECOND independent
        noisy table scores it.
    Report the mean (over `trials`) of each estimator for each action count, i.e. each
    estimator's empirical bias away from the true value 0.

    Args:
        n_actions: iterable[int] — action-count settings to sweep.
        trials:    int — Monte-Carlo trials per setting.
        seed:      int — RNG seed (use numpy default_rng).
    Returns:
        dict {n_act: (single_bias, double_bias)} of floats.
    Note: the two estimate tables must be independent draws.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# Calibrated to the real verify numbers: single(max) bias by #actions(2,4,8) ~ +0.571/+1.029/+1.421,
# double bias ~ 0 (<=0.009). Assert the ROBUST direction with generous slack.
mech = overestimation_mechanism()
biases = [mech[n][0] for n in (2, 4, 8)]
print("single(max) bias by #actions (2,4,8):", [round(b, 3) for b in biases])
print("double bias by #actions:", [round(mech[n][1], 3) for n in (2, 4, 8)])
assert all(b > 0.2 for b in biases), "single(max) estimator must overestimate (bias>0)"
assert biases[0] < biases[1] < biases[2], "overestimation must grow with #actions (Fig.1)"
for n in (2, 4, 8):
    assert abs(mech[n][1]) < 0.05, f"double estimator must be ~unbiased (n={n})"
print("ok")

## Part D. Сборка абляции: обучаем обе руки, меряем эффект

Теперь — реальный прогон. Обучаем vanilla и Double через `train(double=...)` на нескольких сидах (эффект статьи распределительный, один сид — чистый шум), меряем возврат и переоценку на каждую руку, сравниваем с random-полом.

Прогон занимает несколько минут на CPU.

In [ ]:
def run_arm(double, seeds):
    """Train one arm across seeds; return per-seed return and per-seed value-overestimation."""
    rets, overs = [], []
    for sd in seeds:
        online, _, _ = train(double=double, seed=sd)
        m, _ = eval_return(greedy_policy(online), episodes=20)
        _, _, over = value_overestimation(online)
        rets.append(m); overs.append(over)
    return rets, overs

SEEDS = (0, 1, 2)
print(f"device={DEVICE} env={ENV_ID} seeds={SEEDS}")
print("training vanilla DQN (max-operator target) across seeds ...")
rets_dqn, over_dqn = run_arm(False, SEEDS)
print("training Double DQN (decoupled select/evaluate target) across seeds ...")
rets_ddqn, over_ddqn = run_arm(True, SEEDS)

ret_rand = eval_return(random_policy())
mean_dqn, std_dqn = float(np.mean(rets_dqn)), float(np.std(rets_dqn))
mean_ddqn, std_ddqn = float(np.mean(rets_ddqn)), float(np.std(rets_ddqn))
over_dqn_m, over_ddqn_m = float(np.mean(over_dqn)), float(np.mean(over_ddqn))

print(f"\nrandom      return = {ret_rand[0]:.1f} +/- {ret_rand[1]:.1f}")
print(f"vanilla DQN return = {mean_dqn:.1f} +/- {std_dqn:.1f}   per-seed {[round(x,1) for x in rets_dqn]}")
print(f"Double DQN  return = {mean_ddqn:.1f} +/- {std_ddqn:.1f}   per-seed {[round(x,1) for x in rets_ddqn]}")
print(f"\nvalue-overestimation  vanilla = {over_dqn_m:+.2f}   Double = {over_ddqn_m:+.2f}   "
      f"reduction = {over_dqn_m - over_ddqn_m:+.2f}")

In [ ]:
# Ключевая фигура: возврат и переоценка по рукам.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(["random", "vanilla", "Double"], [ret_rand[0], mean_dqn, mean_ddqn],
          yerr=[ret_rand[1], std_dqn, std_ddqn], color=["#999", "#c44", "#39c"], capsize=4)
ax[0].set_title("episode return (higher better)"); ax[0].set_ylabel("return")
ax[1].bar(["vanilla", "Double"], [over_dqn_m, over_ddqn_m], color=["#c44", "#39c"])
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_title("value-overestimation = predicted - actual\n(lower better; paper Fig.3)")
plt.tight_layout(); plt.show()

## Part E. Пощупать фишку — контролируемый контраст

Вот зачёт воспроизведения. Сравнение откалибровано по реальным числам (seeds 0,1,2; 30k шагов CPU): возвраты random=18.8±13.9, vanilla DQN=64.1±43.8 (по сидам 123.9/20.1/48.2), Double DQN=111.5±12.6 (104.3/129.2/101.0); переоценка vanilla=+4.30, Double=−21.45, сокращение=+25.75.

Ассерты ниже проверяют **робастный относительный эффект статьи**, а не тривиальные by-construction-факты:
1. `mean_ddqn >= mean_dqn` — развязка возврат не роняет (держится на всех 3 наборах сидов: (0,1,2), (3,4,5), (10,11,12));
2. `over_ddqn_m < over_dqn_m` — Double строго снижает переоценку ценности относительно vanilla (держится на всех 3 наборах сидов).

Намеренно НЕ гейтим хрупкое: сравнение std между руками и абсолютный знак переоценки vanilla — оба сид-зависимы при таком коротком CPU-бюджете (репортим, но не ассертим). Ни один порог не подгонялся под прохождение — относительное направление эффекта это ровно то, что заявляет статья и что воспроизводится. random-пол оставлен нижним санити-баундом (`ddqn > 3*random`, `dqn > random`).

❓ **Inline.** Посмотри на per-seed возвраты vanilla (123.9/20.1/48.2) против Double (104.3/129.2/101.0). Среднее у Double выше, а разброс по сидам примерно втрое меньше. Какую часть заявки статьи лучше всего видно именно здесь, а не в среднем возврате?

In [ ]:
# baseline floor (lower sanity bound only)
print(f"return  random={ret_rand[0]:.1f}  vanilla={mean_dqn:.1f}  Double={mean_ddqn:.1f}")
assert mean_ddqn > 3 * ret_rand[0], "Double DQN must beat random by a wide margin"
assert mean_dqn > ret_rand[0], "even vanilla DQN must beat random"

# EFFECT (A): decoupling does not hurt return (robust direction, holds on all 3 seed sets).
# std reported but NOT asserted (seed-fragile at this short CPU budget).
print(f"effect  return  Double={mean_ddqn:.1f}+/-{std_ddqn:.1f}  vanilla={mean_dqn:.1f}+/-{std_dqn:.1f}")
assert mean_ddqn >= mean_dqn, \
    "Double DQN mean return must be at least vanilla DQN's (decoupling does not hurt)"

# EFFECT (B): the paper's headline, asserted as the ROBUST RELATIVE CONTRAST.
# Double strictly reduces value-overestimation vs vanilla (holds on all 3 seed sets).
# Absolute sign of vanilla's bias is NOT asserted (seed-dependent, reported only).
print(f"effect  value-overestimation  vanilla={over_dqn_m:+.2f}  Double={over_ddqn_m:+.2f}  "
      f"(reduction={over_dqn_m - over_ddqn_m:+.2f})")
assert over_ddqn_m < over_dqn_m, \
    "Double DQN must reduce value-overestimation below vanilla DQN (van Hasselt 2016)"
print("ok — paper effect reproduced: decoupling preserves return and cuts overestimation")

## Part F. Рефлексия

**Что украсть в работу.** Развязка «кто выбирает» и «кто оценивает» — приём шире RL. Любая оценка вида «возьми лучшее по зашумлённой метрике и им же отчитайся» смещена вверх: выбор top-k по тому же скору, которым потом меряешь качество; early stopping по тому же валидационному сигналу, по которому тюнили. Лечение одинаковое — оценивай выбор на независимом срезе/прогоне.

❓ **Inline (узнавание).** Опиши конкретную НЕ-RL ситуацию из своей практики, где «выбор и оценка по одному и тому же зашумлённому числу» давали оптимистичный сдвиг. Как бы ты развязала их по рецепту Double DQN — какой второй независимый источник оценки взять?

**Что осталось мутным → следующий узел.** Double DQN снимает переоценку от `max`, но не трогает шум исследования, аппроксимацию сетью и off-policy-сдвиг реплея. На больших задачах остаточная переоценка возвращается. Следующие узлы ветки: distributional RL (C51/QR-DQN — моделируем распределение возврата, а не точку), затем dueling/PER. Где, по-твоему, переоценка `max` всё ещё кусается сильнее всего?

---

Если застряла на дырах — раскрой SPOILERS ниже. Сначала попробуй сама: контракты в докстрингах и тесты задают форму ответа однозначно.

## SPOILERS — полные решения

Раскрывай по одному, после серьёзной попытки.

<details><summary>Part A — <code>td_target</code></summary>

```python
def td_target(reward, next_state, done, online, target, gamma, double):
    with torch.no_grad():
        if double:
            next_actions = online(next_state).argmax(dim=1, keepdim=True)   # selection: online
            next_q = target(next_state).gather(1, next_actions)             # evaluation: target
        else:
            next_q = target(next_state).max(dim=1, keepdim=True).values     # max selects AND evaluates
        y = reward + gamma * next_q * (1.0 - done)
    return y
```
</details>

<details><summary>Part B — <code>value_overestimation</code></summary>

```python
@torch.no_grad()
def value_overestimation(net, gamma=0.99, episodes=20, seed=3000):
    pi = greedy_policy(net)
    env = gym.make(ENV_ID)
    pred, actual = [], []
    for i in range(episodes):
        s, _ = env.reset(seed=seed + i)
        v_pred = net(torch.as_tensor(s, dtype=torch.float32)).max().item()
        done, G, disc = False, 0.0, 1.0
        while not done:
            s, r, term, trunc, _ = env.step(pi(s))
            G += disc * r
            disc *= gamma
            done = term or trunc
        pred.append(v_pred); actual.append(G)
    env.close()
    pred_m, act_m = float(np.mean(pred)), float(np.mean(actual))
    return pred_m, act_m, pred_m - act_m
```
</details>

<details><summary>Part B′ — <code>overestimation_gap</code></summary>

```python
@torch.no_grad()
def overestimation_gap(online, target, states):
    q_t = target(states)
    dqn_val = q_t.max(dim=1).values.mean().item()
    a_star = online(states).argmax(dim=1, keepdim=True)
    double_val = q_t.gather(1, a_star).squeeze(1).mean().item()
    return float(dqn_val), float(double_val), float(dqn_val - double_val)
```
</details>

<details><summary>Part C — <code>overestimation_mechanism</code></summary>

```python
def overestimation_mechanism(n_actions=(2, 4, 8), trials=20000, seed=123):
    rng = np.random.default_rng(seed)
    out = {}
    for n in n_actions:
        qa = rng.normal(0.0, 1.0, size=(trials, n))     # estimator A
        qb = rng.normal(0.0, 1.0, size=(trials, n))     # estimator B (independent)
        single = qa.max(axis=1)                          # max selects AND evaluates on A
        double = qb[np.arange(trials), qa.argmax(axis=1)]  # select on A, evaluate on B
        out[n] = (float(single.mean()), float(double.mean()))
    return out
```
</details>